# ✈️ Proyecto de Ingeniería de Datos: Pipeline ETL y Modelado Data Vault 2.0 para Auditoría de Vuelos

Este proyecto forma parte de mi portafolio profesional en **Data Engineering y Analytics**. El objetivo principal es demostrar la capacidad de diseñar, construir y automatizar un flujo de datos (pipeline) escalable y de grado industrial, partiendo desde la extracción de datos crudos hasta la generación de un modelo analítico listo para herramientas de Business Intelligence (BI).

## 📌 Contexto del Negocio
En el sector de la aviación, la puntualidad es crítica. Este proyecto audita un dataset masivo de operaciones aéreas para rastrear, estructurar y analizar las métricas de retrasos en vuelos. Al tener los datos correctamente modelados, el equipo directivo puede identificar cuellos de botella logísticos (por aerolínea o aeropuerto) y tomar decisiones estratégicas fundamentadas.

## 🛠️ Stack Tecnológico y Arquitectura
Para resolver este problema, no nos limitamos a limpiar una simple tabla. Implementamos metodologías avanzadas de ingeniería:
* **Lenguaje Core:** Python 3 (Pandas para procesamiento vectorial).
* **Ingesta Automatizada:** API nativa de Kaggle (`kagglehub`).
* **Arquitectura de Datos:** **Data Vault 2.0**. Utilizamos algoritmos de encriptación (Hashes MD5) para separar los datos en *Hubs* (Entidades), *Links* (Relaciones) y *Satélites* (Contexto). Esto garantiza un modelo 100% auditable, escalable y sin pérdida de historial.
* **Integración BI:** Generación de un esquema normalizado exportado en CSVs listos para construir un modelo de estrella en Power BI o Tableau.

## 🗺️ Hoja de Ruta del Cuaderno
A lo largo de este notebook, se ejecutarán las siguientes fases de forma secuencial:
1. **Aprovisionamiento e Ingesta:** Descarga automática de la fuente de verdad.
2. **Preprocesamiento:** Estandarización de metadatos y limpieza de nulos.
3. **Motor de Transformación:** Generación de llaves subrogadas y construcción del modelo Data Vault 2.0.
4. **Persistencia Física:** Exportación empaquetada de los entregables (ZIP).
5. **Business Intelligence (Bonus):** Análisis descriptivo rápido demostrando la usabilidad del modelo final.

---
*Si deseas ejecutar este código, ve al menú superior de Colab y selecciona **Entorno de ejecución > Ejecutar todo**.*


# 🚀 Fase 1: Inicialización del Entorno y Control de Dependencias

Para construir un pipeline robusto de Ingeniería de Datos, es fundamental asegurar un entorno aislado y reproducible. En esta celda realizamos dos tareas críticas:
1. **Aprovisionamiento de librerías:** Instalamos `kagglehub` para interactuar directamente con la API oficial de Kaggle, eliminando dependencias locales de archivos.
2. **Carga de módulos del core:**
   - `pandas`: Motor de procesamiento vectorial para la manipulación y estructuración de los DataFrames.
   - `hashlib`: Algoritmos criptográficos para generar las llaves subrogadas (MD5) de la arquitectura Data Vault.
   - `zipfile` y `google.colab.files`: Herramientas de infraestructura para empaquetar y descargar los entregables directamente al entorno local del usuario.


In [2]:
!pip install -q kagglehub pandas

import pandas as pd
import os
import glob
import hashlib
import zipfile
from datetime import datetime
from google.colab import files
import kagglehub
import matplotlib.pyplot as plt

print("✅ Librerías importadas correctamente.")

✅ Librerías importadas correctamente.


# 📥 Fase 2: Ingestión de Datos (Data Lineage) desde Kaggle

La automatización de la ingesta de datos directamente desde la API de origen garantiza la **reproducibilidad e idempotencia** del pipeline.
* Utilizaremos la función `dataset_download` de `kagglehub` para consultar, autenticar y descargar la última versión del dataset original sin requerir intervención humana o cargas manuales a Google Drive.
* Al finalizar la descarga, implementamos un método de búsqueda dinámica mediante patrones globales (`glob`) para localizar de manera automática el archivo `.csv` dentro del directorio virtual del sistema operativo de Colab, asignándolo como nuestra fuente única de verdad (`ARCHIVO_LOCAL`).


In [3]:
print("Conectando a Kaggle y descargando el dataset original...")
ORIGEN_DATOS = "KAGGLE_API"

try:
    # Descargar la última versión del dataset directamente desde Kaggle
    ruta_kaggle = kagglehub.dataset_download("patrickzel/flight-delay-and-cancellation-dataset-2019-2023")
    print("Ruta de descarga de Kaggle:", ruta_kaggle)

    # Kaggle descarga una carpeta, así que buscamos el archivo .csv dentro de ella
    archivos_csv = glob.glob(f"{ruta_kaggle}/*.csv")

    if archivos_csv:
        ARCHIVO_LOCAL = archivos_csv[0] # Tomamos el primer CSV que encuentre
        print(f"\n✅ Archivo CSV encontrado y listo para usarse: {ARCHIVO_LOCAL}")
    else:
        print("\n❌ No se encontró ningún archivo CSV en la ruta descargada.")

except Exception as e:
    print(f"Error en la descarga desde Kaggle: {e}")


Conectando a Kaggle y descargando el dataset original...


100%|██████████| 140M/140M [00:01<00:00, 80.3MB/s]

Extracting files...


Ruta de descarga de Kaggle: /root/.cache/kagglehub/datasets/patrickzel/flight-delay-and-cancellation-dataset-2019-2023/versions/7

✅ Archivo CSV encontrado y listo para usarse: /root/.cache/kagglehub/datasets/patrickzel/flight-delay-and-cancellation-dataset-2019-2023/versions/7/flights_sample_3m.csv


# 🔍 Fase 3: Ingestión a Memoria, Estandarización y Preprocesamiento

Antes de estructurar los datos en la arquitectura destino, procesamos los datos crudos bajo estrictos estándares de calidad:
1. **Carga Acotada (Batching):** Limitamos la lectura inicial a `100,000` registros para optimizar los tiempos de ejecución y uso de memoria en nuestro entorno de desarrollo.
2. **Estandarización del Schema (Metadata):** Forzamos todos los nombres de columnas a mayúsculas (`UPPERCASE`) para prevenir inconsistencias por tipografía (case-sensitivity) comunes al integrar fuentes de datos.
3. **Tratamiento de Nulos (Data Cleansing):** Reemplazamos valores nulos (`NaN`) por `0` en las métricas cuantitativas, previniendo excepciones aritméticas y simplificando el posterior análisis en herramientas de BI.

In [4]:
print("Leyendo el archivo CSV...")
# Leemos el archivo detectado en el paso anterior
df_crudo = pd.read_csv(ARCHIVO_LOCAL, nrows=100000)

# Limpieza básica: nombres de columnas a mayúsculas y rellenar nulos
df_crudo.columns = [col.upper() for col in df_crudo.columns]
df_crudo.fillna(0, inplace=True)

print(f"\n¡Éxito! Se cargaron {len(df_crudo):,} registros listos para el modelado.")
display(df_crudo.head(3))

Leyendo el archivo CSV...

¡Éxito! Se cargaron 100,000 registros listos para el modelado.


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,0.0,0.0,0.0,0.0,0.0
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,0.0,0.0,0.0,0.0,0.0
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,0.0,0.0,0.0,0.0,0.0


# 🏗️ Fase 4: Modelado de Datos - Arquitectura Data Vault 2.0

En esta celda transformamos el archivo transaccional plano en una arquitectura **Data Vault 2.0** escalable, auditable e histórica. Esta arquitectura se divide en tres componentes:

### A. Hubs (Claves de Negocio)
Representan las entidades principales del negocio. Son inmutables y contienen:
- La clave de negocio original (ej: Código IATA del Aeropuerto, Código de la Aerolínea).
- Una clave subrogada generada con algoritmos de hashing unidireccionales (**MD5**). Esto desacopla las dependencias del sistema de origen y optimiza las uniones (`JOINs`).
- Metadatos de auditoría: `FECHA_CARGA` y `ORIGEN_RECURSO`.

### B. Links (Relaciones Transaccionales)
Representan la asociación física o transaccional entre los Hubs. En este caso, el `LINK_OPERACION_VUELO` conecta un número de vuelo específico, con su aerolínea operadora, su aeropuerto origen y su destino. Su identificador principal es la combinación hash de todas las claves que lo componen.

### C. Satellites (Contexto e Indicadores de Negocio)
Albergan las propiedades descriptivas de los Links o Hubs que pueden cambiar con el tiempo (métricas cuantitativas, estados, etc.). Nuestro satélite almacena las métricas de retraso asociadas a un Link específico.

In [5]:
print("Iniciando transformación de datos (Generando Hashes MD5)...")

def generar_hash_md5(valor):
    if pd.isna(valor): return None
    return hashlib.md5(str(valor).encode('utf-8')).hexdigest()

# --- DETECCIÓN DINÁMICA DE COLUMNAS ROBUSTA ---
def obtener_columna(opciones):
    for col in opciones:
        if col in df_crudo.columns:
            return col
    return opciones[0] # Retorna la primera por defecto si no encuentra ninguna

col_vuelo = obtener_columna(['FL_NUMBER', 'FLIGHT_NUMBER', 'OP_CARRIER_FL_NUM'])
col_aerolinea = obtener_columna(['AIRLINE', 'OP_UNIQUE_CARRIER'])
col_origen = obtener_columna(['ORIGIN', 'ORIGIN_AIRPORT_ID'])
col_destino = obtener_columna(['DEST', 'DESTINATION', 'DEST_AIRPORT_ID'])
# Muchos datasets usan DEP_DELAY, pero lo cubrimos por si acaso
col_dep_delay = obtener_columna(['DEP_DELAY', 'DEP_DELAY_NEW'])

print(f"Columnas detectadas: Vuelo={col_vuelo}, Aerolínea={col_aerolinea}, Origen={col_origen}, Destino={col_destino}")

ahora = datetime.now().isoformat()

# A. CREAR HUBS
print(" -> Construyendo Hubs...")
aeropuertos = pd.concat([df_crudo[col_origen], df_crudo[col_destino]]).unique()
hub_aeropuerto = pd.DataFrame({'CODIGO_IATA': aeropuertos})
hub_aeropuerto['HK_AEROPUERTO'] = hub_aeropuerto['CODIGO_IATA'].apply(generar_hash_md5)
hub_aeropuerto['FECHA_CARGA'], hub_aeropuerto['ORIGEN_RECURSO'] = ahora, ORIGEN_DATOS

hub_aerolinea = df_crudo[[col_aerolinea]].drop_duplicates().rename(columns={col_aerolinea: 'CODIGO_AEROLINEA'})
hub_aerolinea['HK_AEROLINEA'] = hub_aerolinea['CODIGO_AEROLINEA'].apply(generar_hash_md5)
hub_aerolinea['FECHA_CARGA'], hub_aerolinea['ORIGEN_RECURSO'] = ahora, ORIGEN_DATOS

# B. CREAR LINKS
print(" -> Construyendo Links...")
df_crudo['HK_AEROLINEA'] = df_crudo[col_aerolinea].apply(generar_hash_md5)
df_crudo['HK_AEROPUERTO_ORIGEN'] = df_crudo[col_origen].apply(generar_hash_md5)
df_crudo['HK_AEROPUERTO_DESTINO'] = df_crudo[col_destino].apply(generar_hash_md5)
df_crudo['HK_VUELO'] = df_crudo[col_vuelo].astype(str).apply(generar_hash_md5)

df_crudo['STR_LINK'] = df_crudo[col_vuelo].astype(str) + df_crudo[col_aerolinea].astype(str) + df_crudo[col_origen].astype(str) + df_crudo[col_destino].astype(str)
df_crudo['HK_LINK_OPERACION'] = df_crudo['STR_LINK'].apply(generar_hash_md5)

link_operacion = df_crudo[['HK_LINK_OPERACION', 'HK_VUELO', 'HK_AEROLINEA', 'HK_AEROPUERTO_ORIGEN', 'HK_AEROPUERTO_DESTINO']].drop_duplicates()
link_operacion['FECHA_CARGA'], link_operacion['ORIGEN_RECURSO'] = ahora, ORIGEN_DATOS

# C. CREAR SATELLITES
print(" -> Construyendo Satélites...")
sat_retrasos = df_crudo[['HK_LINK_OPERACION', col_dep_delay]].copy()
sat_retrasos = sat_retrasos.rename(columns={col_dep_delay: 'RETRASO_SALIDA'})
sat_retrasos['FECHA_CARGA'], sat_retrasos['ORIGEN_RECURSO'] = ahora, ORIGEN_DATOS


Iniciando transformación de datos (Generando Hashes MD5)...
Columnas detectadas: Vuelo=FL_NUMBER, Aerolínea=AIRLINE, Origen=ORIGIN, Destino=DEST
 -> Construyendo Hubs...
 -> Construyendo Links...
 -> Construyendo Satélites...


# 💾 Fase 5: Exportación a CSV y Empaquetado de Entregables (ZIP)

En esta celda final, consolidamos el proceso de ingeniería y preparamos las salidas físicas del pipeline para su uso directo en plataformas analíticas de Business Intelligence como **Power BI** o **Tableau**:

1. **Persistencia Física (CSV):** Exportamos cada componente del modelo (Hubs, Links y Satélites) en formatos planos separados. Al mantenerlos normalizados e independientes, evitamos la redundancia de datos (duplicados innecesarios) propia de un CSV masivo de 600 MB.
2. **Empaquetado (ZIP Archive):** Comprimimos los 4 archivos CSV individuales en un solo contenedor unificado (`datos_datavault.zip`). Esto optimiza la transferencia de red y la organización del espacio de almacenamiento.
3. **Trigger de Descarga Automática:** Usamos las APIs nativas del navegador de Google Colab para transferir directamente el paquete resultante a la computadora local del analista, listo para modelarse visualmente.


In [6]:
print("Exportando datos a archivos CSV...")

tablas = {
    'HUB_AEROPUERTO': hub_aeropuerto,
    'HUB_AEROLINEA': hub_aerolinea,
    'LINK_OPERACION_VUELO': link_operacion,
    'SAT_RETRASOS_VUELO': sat_retrasos
}

archivos = []
for nombre, df_tabla in tablas.items():
    archivo_csv = f"{nombre}.csv"
    df_tabla.to_csv(archivo_csv, index=False)
    archivos.append(archivo_csv)
    print(f"  [OK] Tabla {nombre} exportada con éxito.")

# Comprimir en un ZIP
with zipfile.ZipFile('datos_datavault.zip', 'w') as zipf:
    for archivo in archivos:
        zipf.write(archivo)

print("\n✅ [PROCESO TERMINADO] Descargando 'datos_datavault.zip' a tu computadora...")
files.download('datos_datavault.zip')


Exportando datos a archivos CSV...
  [OK] Tabla HUB_AEROPUERTO exportada con éxito.
  [OK] Tabla HUB_AEROLINEA exportada con éxito.
  [OK] Tabla LINK_OPERACION_VUELO exportada con éxito.
  [OK] Tabla SAT_RETRASOS_VUELO exportada con éxito.

✅ [PROCESO TERMINADO] Descargando 'datos_datavault.zip' a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 📈 Fase 6: Dashboard Interactivo (Analytics Dinámico)

Para elevar la calidad del entregable, utilizaremos **Plotly Express**. A diferencia de las gráficas estáticas, este dashboard permite:
1. **Exploración Táctil:** Hacer zoom en rangos específicos de tiempo o retraso.
2. **Tooltips Informativos:** Ver el valor exacto de cada barra o punto al pasar el mouse.
3. **Filtros Dinámicos:** Interactuar con la leyenda para aislar datos específicos.

In [8]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Preparar datos para el dashboard: Unir con Hub de Aerolíneas para tener nombres
df_dashboard = pd.merge(df_analisis, hub_aerolinea, on='HK_AEROLINEA')

# 2. Gráfico 1: Top Aerolíneas con mayor promedio de retraso
retraso_por_aerolinea = df_dashboard[df_dashboard['RETRASO_SALIDA'] > 0] \
    .groupby('CODIGO_AEROLINEA')['RETRASO_SALIDA'].mean().reset_index() \
    .sort_values(by='RETRASO_SALIDA', ascending=False)

fig_bar = px.bar(
    retraso_por_aerolinea,
    x='CODIGO_AEROLINEA',
    y='RETRASO_SALIDA',
    title='Promedio de Retraso por Aerolínea (Minutos)',
    labels={'RETRASO_SALIDA': 'Minutos Promedio', 'CODIGO_AEROLINEA': 'Aerolínea'},
    color='RETRASO_SALIDA',
    color_continuous_scale='Reds'
)

fig_bar.update_layout(template='plotly_white')
fig_bar.show()

# 3. Gráfico 2: Distribución Interactiva de Retrasos
fig_hist = px.histogram(
    df_dashboard[df_dashboard['RETRASO_SALIDA'] > 0],
    x='RETRASO_SALIDA',
    nbins=100,
    title='Distribución Detallada de Retrasos',
    labels={'RETRASO_SALIDA': 'Minutos de Retraso'},
    marginal='box', # Añade un diagrama de caja superior para ver outliers
    color_discrete_sequence=['#3498db']
)

fig_hist.update_layout(xaxis_range=[0, 200], template='plotly_white')
fig_hist.show()

## 📊 Fase 8: Master Dashboard Interactivo (5 Paneles de Control)

En esta sección centralizamos 5 métricas clave para una auditoría completa de las operaciones aéreas.

In [12]:
import plotly.express as px
import pandas as pd

# Preparación de datos maestros
df_full = pd.merge(df_analisis, hub_aerolinea, on='HK_AEROLINEA')
df_positives = df_full[df_full['RETRASO_SALIDA'] > 0]

# 1. Dashboard de Rendimiento por Aerolínea (Barras)
fig1 = px.bar(
    df_positives.groupby('CODIGO_AEROLINEA')['RETRASO_SALIDA'].mean().reset_index().sort_values('RETRASO_SALIDA'),
    x='RETRASO_SALIDA', y='CODIGO_AEROLINEA', orientation='h',
    title='1. Ranking de Impuntualidad (Promedio Minutos)',
    color='RETRASO_SALIDA', color_continuous_scale='Viridis'
)
fig1.show()

# 2. Análisis de Outliers y Distribución (Box + Hist)
fig2 = px.histogram(
    df_positives, x='RETRASO_SALIDA', marginal='box',
    title='2. Distribución y Valores Atípicos de Retrasos',
    labels={'RETRASO_SALIDA': 'Minutos'},
    color_discrete_sequence=['#e74c3c']
)
fig2.update_layout(xaxis_range=[0, 150])
fig2.show()

# 3. Top 10 Aeropuertos de Origen con Mayores Retrasos (TreeMap)
# Usamos suffixes para evitar el MergeError con columnas de auditoría
df_airports = pd.merge(df_positives, hub_aeropuerto, left_on='HK_AEROPUERTO_ORIGEN', right_on='HK_AEROPUERTO', suffixes=('_VUELO', '_HUB'))
top_airports = df_airports.groupby('CODIGO_IATA')['RETRASO_SALIDA'].mean().nlargest(10).reset_index()

fig3 = px.treemap(
    top_airports, path=['CODIGO_IATA'], values='RETRASO_SALIDA',
    title='3. Top 10 Aeropuertos Críticos (Concentración de Demoras)',
    color='RETRASO_SALIDA', color_continuous_scale='Blues'
)
fig3.show()

# 4. Relación Vuelo vs Retraso (Scatter)
fig4 = px.scatter(
    df_positives.sample(min(1000, len(df_positives))),
    x='HK_VUELO', y='RETRASO_SALIDA', color='CODIGO_AEROLINEA',
    title='4. Dispersión de Retrasos por ID de Vuelo (Muestra)',
    hover_data=['RETRASO_SALIDA']
)
fig4.update_layout(xaxis_showticklabels=False)
fig4.show()

# 5. Participación de Mercado en Retrasos (Donut Chart)
fig5 = px.pie(
    df_full, names='CODIGO_AEROLINEA',
    title='5. % de Operaciones Totales por Aerolínea',
    hole=0.4
)
fig5.update_traces(textinfo='percent+label')
fig5.show()